# Final Model Training & Evaluation Notebook

**Purpose of the Notebook**

Objective

The goal of this notebook is to **train, evaluate, and compare** three different EEG representations for predicting **unaided brand recall** at the *ad segment level*:

1. **Handcrafted (classic) EEG features** – Traditional frequency-domain and time-domain features
2. **TS2Vec embeddings** – Self-supervised contrastive learning representations
3. **FEMBA embeddings** – Foundation model-based representations for EEG

All models are evaluated using a **fixed, leakage-safe data split** and a **common baseline classifier (Gradient Boosting)** to ensure fair comparison.


This comparative framework allows us to assess the trade-offs between:
- **Predictive performance** – Which representation yields the best predictions?
- **Interpretability** – Can we understand what drives the predictions?
- **Computational cost** – What are the resource requirements?
- **Generalization** – How well do models perform on unseen data?

---

## 1. Imports & Setup

In [59]:
# Core libraries
import numpy as np
import pandas as pd
import joblib
import time
import json
from pathlib import Path

# Machine learning
from sklearn.ensemble import AdaBoostClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    brier_score_loss, roc_curve, auc
)
from sklearn.calibration import calibration_curve

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Global Configuration
RANDOM_STATE = 22122025
np.random.seed(RANDOM_STATE)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"✓ Random seed set to: {RANDOM_STATE}")

✓ Random seed set to: 22122025


In [60]:
# Model output path - Where models will be saved
MODEL_OUTPUT_PATH = Path("../../models/")

# Create output directory if it doesn't exist
MODEL_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

## 2. Load Data 


**Data Loading**

We load three different EEG representations, all prepared at the **ad segment level**:

1. **average EEG features** – Handcrafted frequency-domain and statistical features
2. **TS2Vec embeddings** – Self-supervised contrastive learning representations
3. **FEMBA embeddings** – Foundation model embeddings

All datasets are **aligned on the same observations**, ensuring that each row across all three datasets corresponds to the same ad segment viewing event.

**Precomputed Train-Test Split**

To ensure fair comparison and prevent data leakage:

> **The same train–test split is reused across all three datasets.** This split was created using participant-level stratification to avoid information leakage between train and test sets.


In [61]:
# Load train and val data
train_df = pd.read_parquet('../../data/final_for_modeling/train_ad_break_level_data.parquet')
test_df = pd.read_parquet('../../data/final_for_modeling/test_ad_break_level_data.parquet')

segment_df = pd.read_parquet('../../data/final_for_modeling/segment_df.parquet')
ts2vec_df = pd.read_parquet('../../data/final_for_modeling/bin_ts2vec_emb_df.parquet')
femba_df = pd.read_parquet('../../data/final_for_modeling/bin_femba_emb_df.parquet')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"-----------------------------------")
print(f"Segment shape: {segment_df.shape}")
print(f"TS2Vec Emb shape: {ts2vec_df.shape}")
print(f"FEMBA Emb shape: {femba_df.shape}")

Train shape: (1970, 22)
Test shape: (865, 22)
-----------------------------------
Segment shape: (2835, 50)
TS2Vec Emb shape: (2835, 129)
FEMBA Emb shape: (2835, 129)


In [62]:
target_col = 'unaided_brand_recall'

drop_cols = [
    'ad_break_session_key', 'participant_id_calculated',
]

dummy_columns = ['category', 'brand',  'content_watched', 
                'device', 'seen_ad_before_score', 'age_group', 'gender']

def custom_get_dummies(df, columns = dummy_columns):
    """Custom one-hot encoding for categorical variables."""
    final_df = df.copy()
    final_df = pd.get_dummies(final_df, columns=columns)
    return final_df

def custom_load(df, train_df=train_df, test_df=test_df):
    """
    Custom function to load and preprocess the data for modeling.
    
    outputs:
        X_train_df, X_test_df, y_train, y_test
    """
    
    # make the big dataframe
    df_train = train_df.merge(df, on='ad_break_session_key', how='left')
    df_test = test_df.merge(df, on='ad_break_session_key', how='left')
    
    # TEMP Drop null rows
    df_train = df_train.dropna(axis=0)
    df_test = df_test.dropna(axis=0)
    
    # dummy encoding
    df_train = custom_get_dummies(df_train).drop(columns='seen_ad_before_score_Unknown')
    df_test = custom_get_dummies(df_test).drop(columns='seen_ad_before_score_Unknown')
    
    # drop key columns
    df_train = df_train.drop(columns=drop_cols)
    df_test = df_test.drop(columns=drop_cols)
    
    # split into X and y
    X_train_df = df_train.drop(columns=[target_col])
    y_train = df_train[target_col]
    
    X_test_df = df_test.drop(columns=[target_col])
    y_test = df_test[target_col]
    
    print(f" Test shape: {df_test.shape}, Train shape: {df_train.shape}")
    print(f"-----------------------------------")
    
    return X_train_df, X_test_df, y_train, y_test

In [63]:
# Load classic mean with Custom load
X_classic_mean_train, X_classic_mean_test, y_classic_mean_train, yclassic_mean_test = custom_load(segment_df)

# Ts2Vec Embeddings
X_ts2vec_train, X_ts2vec_test, y_ts2vec_train, y_ts2vec_test = custom_load(ts2vec_df)

# FEMBA Embeddings
X_femba_train, X_femba_test, y_femba_train, y_femba_test = custom_load(femba_df)


# create a dictionary to hold all datasets
datasets_info = {
    "classic_mean": {
        "X_train": X_classic_mean_train,
        "X_test": X_classic_mean_test,
        "y_train": y_classic_mean_train,
        "y_test": yclassic_mean_test
    },
    "ts2vec": {
        "X_train": X_ts2vec_train,
        "X_test": X_ts2vec_test,
        "y_train": y_ts2vec_train,
        "y_test": y_ts2vec_test
    },
    "femba": {
        "X_train": X_femba_train,
        "X_test": X_femba_test,
        "y_train": y_femba_train,
        "y_test": y_femba_test
    }
}

 Test shape: (865, 145), Train shape: (1970, 145)
-----------------------------------
 Test shape: (865, 224), Train shape: (1970, 224)
-----------------------------------
 Test shape: (865, 224), Train shape: (1970, 224)
-----------------------------------


## 3. Load Benchmark Model

In [ ]:
benchmark_model_path = MODEL_OUTPUT_PATH / "baseline_models/benchmark_model.pkl"
benchmark_model = joblib.load(benchmark_model_path)

## 4. Training Phase (Per Dataset)

In [ ]:
# Storage for all results
results = {
    "classic_mean": {},
    "ts2vec": {},
    "femba": {}
}

for name in ["classic_mean", "ts2vec", "femba"]:
    print(f"\n{'='*60}")
    print(f"Training on {name.upper()} representation")
    print(f"{'='*60}")
    
    # Get pre-loaded data (already preprocessed)
    X_train = datasets_info[name]["X_train"]
    X_test = datasets_info[name]["X_test"]
    y_train = datasets_info[name]["y_train"]
    y_test = datasets_info[name]["y_test"]
    
    # Train and time
    print(f"\n⏱️  Training model started...")
    start_time = time.time()
    benchmark_model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    print(f"✓ Training completed in {train_time:.2f} seconds")
    
    # Generate predictions
    y_pred_test = benchmark_model.predict(X_test)
    y_proba_test = benchmark_model.predict_proba(X_test)[:, 1]
    
    # Also get training predictions for overfitting check
    y_pred_train = benchmark_model.predict(X_train)
    y_proba_train = benchmark_model.predict_proba(X_train)[:, 1]
    
    # Store results
    results[name] = {
        "model": benchmark_model,
        "train_time": train_time,
        "n_features": X_train.shape[1],
        # Test set
        "y_test": y_test,
        "y_pred_test": y_pred_test,
        "y_proba_test": y_proba_test,
        # Train set (for overfitting analysis)
        "y_train": y_train,
        "y_pred_train": y_pred_train,
        "y_proba_train": y_proba_train,
    }
    
    print(f"✓ Predictions generated")
    print(f"✓ Results stored")

print("\n" + "=" * 60)
print("ALL MODELS TRAINED SUCCESSFULLY")
print("=" * 60)


Training on CLASSIC_MEAN representation

⏱️  Training model started...
✓ Training completed in 3.62 seconds
✓ Predictions generated
✓ Results stored

Training on TS2VEC representation

⏱️  Training model started...
✓ Training completed in 8.54 seconds
✓ Predictions generated
✓ Results stored

Training on FEMBA representation

⏱️  Training model started...
✓ Training completed in 8.77 seconds
✓ Predictions generated
✓ Results stored

ALL MODELS TRAINED SUCCESSFULLY


## 5. Evaluation & Results  

#### 5.1 Performance Metrics

**Primary Metrics:**

- **ROC-AUC** → Main comparison metric (ranking quality)
- **F1-score** → Harmonic mean of precision and recall (imbalance-aware)
- **Recall (Sensitivity)** → Ability to detect positive cases (brand recall)
- **Precision** → Reliability of positive predictions

These metrics complement each other and provide a complete picture of model performance.

In [66]:
metrics_summary = {}

for name in ["classic_mean", "ts2vec", "femba"]:
    # Extract predictions
    y_test = results[name]["y_test"]
    y_pred = results[name]["y_pred_test"]
    y_proba = results[name]["y_proba_test"]
    
    # Calculate metrics
    roc_auc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    
    # Store metrics
    metrics_summary[name] = {
        "roc_auc": roc_auc,
        "f1_score": f1,
        "recall": recall,
        "precision": precision,
        "accuracy": (y_test == y_pred).mean()
    }
    

metrics_summary = pd.DataFrame(metrics_summary)
print("\nModel Performance Summary:")
print(metrics_summary.T.to_string(float_format="{:.4f}".format, justify="center"))


Model Performance Summary:
              roc_auc  f1_score  recall  precision  accuracy
classic_mean  0.8981    0.8774   0.9390   0.8233     0.8358 
ts2vec        0.8930    0.8853   0.9556   0.8246     0.8451 
femba         0.8939    0.8755   0.9427   0.8173     0.8324 


#### 5.2 Stability & Generalization

**Overfitting Assessment:**

Large performance gaps between training and test sets indicate overfitting:
- Model memorizes training data
- Poor generalization to new data
- Less reliable in production

We compare train vs. test performance to assess model stability.

In [67]:
for name in ["classic_mean", "ts2vec", "femba"]:
    print(f"\n{name.upper()}:")
    print("-" * 40)
    
    # Train metrics
    y_train = results[name]["y_train"]
    y_proba_train = results[name]["y_proba_train"]
    roc_auc_train = roc_auc_score(y_train, y_proba_train)
    
    # Test metrics
    y_test = results[name]["y_test"]
    y_proba_test = results[name]["y_proba_test"]
    roc_auc_test = roc_auc_score(y_test, y_proba_test)
    
    # Calculate gap
    gap = roc_auc_train - roc_auc_test
    
    # Store
    metrics_summary.loc["roc_auc_train", name] = roc_auc_train
    metrics_summary.loc["roc_auc_gap", name] = gap
    
    print(f"  Train ROC-AUC: {roc_auc_train:.4f}")
    print(f"  Test ROC-AUC:  {roc_auc_test:.4f}")
    print(f"  Gap:           {gap:.4f}", end="")
    
    if gap > 0.15:
        print(" ⚠️  (High overfitting)")
    elif gap > 0.05:
        print(" ⚡ (Moderate overfitting)")
    else:
        print(" ✓ (Good generalization)")

print("\n" + "=" * 60)


CLASSIC_MEAN:
----------------------------------------
  Train ROC-AUC: 0.9683
  Test ROC-AUC:  0.8981
  Gap:           0.0702 ⚡ (Moderate overfitting)

TS2VEC:
----------------------------------------
  Train ROC-AUC: 0.9802
  Test ROC-AUC:  0.8930
  Gap:           0.0872 ⚡ (Moderate overfitting)

FEMBA:
----------------------------------------
  Train ROC-AUC: 0.9759
  Test ROC-AUC:  0.8939
  Gap:           0.0820 ⚡ (Moderate overfitting)



#### 5.3 Interpretability

**Classic EEG Features:**
- **High interpretability** – Feature importance reveals which brain patterns drive predictions
- Can link to neurophysiological theory (e.g., engagement, attention, asymmetry)
- Group-level analysis possible (EI, AI, FAA, PSD bands)

**Embedding-Based Representations (TS2Vec, FEMBA):**
- **Low interpretability** – Latent representations lack direct meaning
- Trade-off: Higher predictive power vs. reduced explainability
- Possible approaches: Permutation importance, dimensionality reduction visualization

We compute feature importance for the classic model to demonstrate interpretability.

In [68]:
# Store interpretability assessment
metrics_summary.loc["interpretability", "classic_mean"] = "High"
metrics_summary.loc["interpretability", "ts2vec"] = "Low"
metrics_summary.loc["interpretability", "femba"] = "Low"

#### 5.4 Computational Cost

**Why This Matters:**

Representation learning methods (TS2Vec, FEMBA) introduce computational overhead:
- Embedding extraction cost
- Higher-dimensional features
- Increased memory requirements

For practical deployment, we must consider:
- Training time
- Feature dimensionality
- Preprocessing overhead

This analysis quantifies the trade-off between accuracy and computational efficiency.

In [69]:
comp_cost = []

for name in ["classic_mean", "ts2vec", "femba"]:
    train_time = results[name]["train_time"]
    n_features = results[name]["n_features"]
    
    metrics_summary.loc["train_time_sec", name] = train_time
    
    comp_cost.append({
        "Representation": name.upper(),
        "Training Time (sec)": f"{train_time:.2f}",
        "Number of Features": n_features,
    })
    
    print(f"\n{name.upper()}:")
    print(f"  Training Time:  {train_time:.2f} seconds")
    print(f"  # Features:     {n_features}")

print("\n" + "=" * 60)

# Create comparison table
comp_df = pd.DataFrame(comp_cost)
print("\n", comp_df.to_string(index=False, justify="center"))


CLASSIC_MEAN:
  Training Time:  3.62 seconds
  # Features:     144

TS2VEC:
  Training Time:  8.54 seconds
  # Features:     223

FEMBA:
  Training Time:  8.77 seconds
  # Features:     223


 Representation Training Time (sec)  Number of Features
 CLASSIC_MEAN          3.62                144        
       TS2VEC          8.54                223        
        FEMBA          8.77                223        


Computational Time Report

> This is taken from the next python notebook file, used to sampleing the time.

In [70]:
# Load Training time for embeddings and training the models
embedding_time_report_path = r'../../data/embedding_time_report.csv'
training_time_report_path = r'../../data/model_training_report.csv'

embedding_time_df = pd.read_csv(embedding_time_report_path)
training_time_df = pd.read_csv(training_time_report_path)

print("\nEmbedding Time Report:")
print(embedding_time_df.to_string(index=False, justify="center"))
print("\nModel Training Time Report:")
print(training_time_df.to_string(index=False, justify="center"))


Embedding Time Report:
 sample  fit_time_femba  encode_time_femba  total_time_s_femba  fit_time_ts2vec  encode_time_ts2vec  total_time_s_ts2vec
    1        23.314           0.113               23.428           393.217             1.826              395.043       
    2        21.736           0.132               21.868           367.386             1.470              368.857       
    3        23.063           0.086               23.149           365.669             1.507              367.176       
    4        23.204           0.104               23.308           359.444             2.288              361.732       
    5        21.639           0.087               21.726           359.005             1.685              360.690       
    6        23.142           0.094               23.236           356.759             1.453              358.212       
    7        22.234           0.156               22.390           373.370             2.511              375.882       
    8   

In [71]:
# create confidence intervals for the training times

report_df = pd.DataFrame()

for name in ["ts2vec", "femba", "classic_mean"]:
    report_df.loc[name, 'name'] = name
    if name == 'femba':
        report_df.loc[name, 'mean_embedding_time'] = embedding_time_df['total_time_s_femba'].mean()
        report_df.loc[name, 'std_embedding_time'] = embedding_time_df['total_time_s_femba'].std()
        
    if name == 'ts2vec':
        report_df.loc[name, 'mean_embedding_time'] = embedding_time_df['total_time_s_ts2vec'].mean()
        report_df.loc[name, 'std_embedding_time'] = embedding_time_df['total_time_s_ts2vec'].std()
    
    report_df.loc[name, 'mean_training_time'] = training_time_df[f'{name}_train_time'].mean()
    report_df.loc[name, 'std_training_time'] = training_time_df[f'{name}_train_time'].std()
    
print(report_df.to_string(float_format="{:.2f}".format, justify="center", index=False))

# calculate 95% confidence intervals
report_df['embedding_time_ci_lower'] = report_df['mean_embedding_time'] - 1.96 * (report_df['std_embedding_time'] / np.sqrt(len(embedding_time_df)))
report_df['embedding_time_ci_upper'] = report_df['mean_embedding_time'] + 1.96 * (report_df['std_embedding_time'] / np.sqrt(len(embedding_time_df)))
report_df['training_time_ci_lower'] = report_df['mean_training_time'] - 1.96 * (report_df['std_training_time'] / np.sqrt(len(training_time_df)))
report_df['training_time_ci_upper'] = report_df['mean_training_time'] + 1.96 * (report_df['std_training_time'] / np.sqrt(len(training_time_df)))

print("\n" + "=" * 60)
print("Computation Time Report with 95% Confidence Intervals:")
for index, row in report_df.iterrows():
    print(f"\n{row['name'].upper()}:")
    if row['name'] != 'classic_mean':
        print(f"  Embedding Time: {row['mean_embedding_time']:.1f} sec (95% CI: {row['embedding_time_ci_lower']:.1f} - {row['embedding_time_ci_upper']:.1f})")
    print(f"  Training Time:  {row['mean_training_time']:.1f} sec (95% CI: {row['training_time_ci_lower']:.1f} - {row['training_time_ci_upper']:.1f})")
    
print("\n" + "=" * 60)


    name      mean_embedding_time  std_embedding_time  mean_training_time  std_training_time
      ts2vec        368.93               10.71               15.48               0.36       
       femba         22.83                0.62               15.34               0.25       
classic_mean           NaN                 NaN                6.33               0.31       

Computation Time Report with 95% Confidence Intervals:

TS2VEC:
  Embedding Time: 368.9 sec (95% CI: 362.3 - 375.6)
  Training Time:  15.5 sec (95% CI: 15.3 - 15.7)

FEMBA:
  Embedding Time: 22.8 sec (95% CI: 22.4 - 23.2)
  Training Time:  15.3 sec (95% CI: 15.2 - 15.5)

CLASSIC_MEAN:
  Training Time:  6.3 sec (95% CI: 6.1 - 6.5)



## 6. Comparative Summary Table

In [72]:
# Display the table
print("\n" + "=" * 80)
print("COMPREHENSIVE COMPARISON TABLE")
print("=" * 80)
print(metrics_summary.T.to_string(justify="center", float_format="{:.4f}".format))
print("\n" + "=" * 80)


print("\n" + "=" * 60)
print("Computation Time Report with 95% Confidence Intervals:")
for index, row in report_df.iterrows():
    print(f"\n{row['name'].upper()}:")
    if row['name'] != 'classic_mean':
        print(f"  Embedding Time: {row['mean_embedding_time']:.1f} sec (95% CI: {row['embedding_time_ci_lower']:.1f} - {row['embedding_time_ci_upper']:.1f})")
    print(f"  Training Time:  {row['mean_training_time']:.1f} sec (95% CI: {row['training_time_ci_lower']:.1f} - {row['training_time_ci_upper']:.1f})")
    
print("\n" + "=" * 60)



COMPREHENSIVE COMPARISON TABLE
             roc_auc f1_score recall precision accuracy roc_auc_train roc_auc_gap interpretability train_time_sec
classic_mean  0.8981  0.8774  0.9390   0.8233   0.8358      0.9683       0.0702         High           3.6233    
ts2vec        0.8930  0.8853  0.9556   0.8246   0.8451      0.9802       0.0872          Low           8.5388    
femba         0.8939  0.8755  0.9427   0.8173   0.8324      0.9759       0.0820          Low           8.7726    


Computation Time Report with 95% Confidence Intervals:

TS2VEC:
  Embedding Time: 368.9 sec (95% CI: 362.3 - 375.6)
  Training Time:  15.5 sec (95% CI: 15.3 - 15.7)

FEMBA:
  Embedding Time: 22.8 sec (95% CI: 22.4 - 23.2)
  Training Time:  15.3 sec (95% CI: 15.2 - 15.5)

CLASSIC_MEAN:
  Training Time:  6.3 sec (95% CI: 6.1 - 6.5)

